# Confidence calibration

How well the model's stated `confidence` matches how often it is right, for one `evaluate` sweep.

After https://arxiv.org/pdf/2512.16030

In [49]:
%matplotlib ipympl

In [50]:
import pandas as pd

from analysis.build_data import load_trace_df

In [51]:
# one row per forecast of every child run of the sweep (see analysis/build_data.py)
forecasts = load_trace_df("outcome-confidence")

## Expected calibration error

ECE (Naeini et al., 2015) is the average gap between confidence and accuracy, with forecasts binned by their stated confidence into $B$ bins:

$$\mathrm{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{N}\,\bigl|\mathrm{acc}(B_b) - \mathrm{conf}(B_b)\bigr|$$

Here a forecast is the outcome prompts' yes/no call, `confidence` is the model's stated probability that the call is right, and a forecast is accurate when the call matches `outcome_occurred`.

In [52]:
# scoreable forecasts: a yes/no call with a confidence, on a market that has resolved
calibrated = forecasts.dropna(
    subset=["predicted_outcome", "confidence", "outcome_occurred"]
)

In [53]:
calibrated["correct"] = calibrated["predicted_outcome"] == calibrated[
    "outcome_occurred"
].astype(bool)

C:\Users\andre\AppData\Local\Temp\ipykernel_28900\3622577461.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  calibrated["correct"] = calibrated["predicted_outcome"] == calibrated[


In [54]:
B = 10

In [55]:
confidence_bins = pd.IntervalIndex.from_breaks([b / B for b in range(B + 1)])

In [56]:
calibrated["confidence_bin"] = pd.cut(
    calibrated["confidence"], confidence_bins, include_lowest=True
)

C:\Users\andre\AppData\Local\Temp\ipykernel_28900\4195307085.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  calibrated["confidence_bin"] = pd.cut(


In [57]:
def expected_calibration_error(forecasts: pd.DataFrame) -> float:
    """sum_b |B_b|/N * |acc(B_b) - conf(B_b)| over the confidence bins."""
    bins = forecasts.groupby("confidence_bin", observed=True)
    acc = bins["correct"].mean()
    conf = bins["confidence"].mean()
    weight = bins.size() / len(forecasts)
    return (weight * (acc - conf).abs()).sum()

In [58]:
ece = expected_calibration_error(calibrated)

In [59]:
ece_by_model = calibrated.groupby("model").apply(
    expected_calibration_error, include_groups=False
)

In [60]:
accuracy_by_model = calibrated.groupby("model")["correct"].mean()

In [61]:
# per model: how often its call was right, and how far its confidence is from that
model_table = pd.DataFrame({"accuracy": accuracy_by_model, "ece": ece_by_model})

In [62]:
model_table

,accuracy,ece
model,,
claude-haiku-4-5,0.632258,0.101677
claude-opus-5-5,0.709677,0.125432


In [ ]:
# the reliability table behind the number: per bin, its size, conf(B_b) and acc(B_b)
reliability = calibrated.groupby("confidence_bin", observed=True).agg(
    n=("correct", "size"), conf=("confidence", "mean"), acc=("correct", "mean")
)